In [2]:
import xarray as xr
import xeofs as xe
import matplotlib.pyplot as plt


dataset='CMF_low.nc' 

#choose between: 
# 'COT_liq.nc' or    (Cloud Optical Thickness of liqiud clouds)
# 'CMF_low.nc' or    (Cloud Mask Fraction of low clouds) 
# 'CPS_liq.nc'       (Cloud Particle Size of liquid cloud)
#or create a new file with concat.ipynb and choose that one

# --- Laden ---
ds = xr.open_dataset('/projekt1/ag_maahn/data_obs_nobackup/modis/MCD06COSP_D3/merged/'+dataset)
print(ds) # Variablennamen checken, dann ggf. anpassen:
cloud = ds["Mean"]  

# --- EOF ---
model = xe.single.EOF(n_modes=6, use_coslat=True)
model.fit(cloud, dim="time")

eofs   = model.components()               # [mode, lat, lon]
pcs    = model.scores()                   # [time, mode]
expvar = model.explained_variance_ratio() # Anteil pro Mode

# --- Scree-Plot ---
fig, ax = plt.subplots()
ax.bar(range(1, 7), expvar.values * 100)
ax.set_xlabel("EOF Mode")
ax.set_ylabel("Erklärte Varianz (%)")
ax.set_title("Scree Plot")
plt.tight_layout()
plt.savefig("scree.png", dpi=150)

# --- Räumliche Muster ---
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for i, ax in enumerate(axes.flat):
    eofs.sel(mode=i+1).plot(ax=ax, cmap="RdBu_r", center=0)
    ax.set_title(f"EOF {i+1}  ({expvar.values[i]*100:.1f}%)")
plt.tight_layout()
plt.savefig("eofs_spatial.png", dpi=150)

# --- PC-Zeitreihen ---
fig, axes = plt.subplots(6, 1, figsize=(14, 12), sharex=True)
for i, ax in enumerate(axes):
    pcs.sel(mode=i+1).plot(ax=ax, lw=0.7)
    ax.set_title(f"PC {i+1}", fontsize=9)
    ax.axhline(0, color="k", lw=0.5)
plt.tight_layout()
plt.savefig("pcs_timeseries.png", dpi=150)

plt.show()

<xarray.Dataset> Size: 414MB
Dimensions:             (time: 8621, longitude: 40, latitude: 30)
Coordinates:
  * time                (time) datetime64[ns] 69kB 2002-07-04 ... 2026-03-11
  * longitude           (longitude) float64 320B -79.5 -78.5 ... -41.5 -40.5
  * latitude            (latitude) float64 240B -19.5 -18.5 -17.5 ... 8.5 9.5
Data variables:
    Mean                (time, longitude, latitude) float64 83MB ...
    Standard_Deviation  (time, longitude, latitude) float64 83MB ...
    Sum                 (time, longitude, latitude) float64 83MB ...
    Pixel_Counts        (time, longitude, latitude) float64 83MB ...
    Sum_Squares         (time, longitude, latitude) float64 83MB ...
Attributes:
    long_name:     Cloud Fraction from Cloud Mask (Low Clouds, CTP GE 680 hPa...
    units:         none
    _FillValue:    -999.0
    valid_min:     0.0
    valid_max:     1.0
    scale_factor:  1.0
    add_offset:    0.0


/home/oscholz/miniforge3/envs/modis_env/lib/python3.11/site-packages/xeofs/preprocessing/multi_index_converter.py:49: FutureWarning: Deleting a single level of a MultiIndex is deprecated. Previously, this deleted all levels of a MultiIndex. Please also drop the following variables: {'dim2', 'dim1'} to avoid an error in the future.
  X_transformed = X_transformed.drop_vars(dim)


ValueError: Input data contains partial NaN entries, which will cause the the SVD to fail.

In [ ]:
import cdsapi
import os

output_dir = "/projekt1/ag_maahn/wind_reanalysis/"

dataset = "reanalysis-era5-pressure-levels"
request = {
    "product_type": ["reanalysis"],
    "variable": [
        "u_component_of_wind",
        "v_component_of_wind"
    ],
    "year": [
        "2014", "2015", "2016",
        "2017", "2018", "2019",
        "2020", "2021", "2022",
        "2023", "2024"
    ],
    "month": ["01","02","03","04","05","06","07","08","09","10","11","12"],
    "day": [f"{d:02d}" for d in range(1, 32)],
    "time": ["12:00"],
    "pressure_level": ["1000"],
    "data_format": "netcdf",
    "download_format": "unarchived",
    "area": [10, -80, -20, -40]
}

client = cdsapi.Client()

target_file = os.path.join(output_dir, "era5_amazon_wind_1000hPa_2014_2024.nc")

client.retrieve(dataset, request).download(target=target_file)